In [2]:
# Llama-3 ko efficiently chalane ke liye quantization libraries
!pip install -q -U bitsandbytes accelerate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 121.6 MB/s eta 0:00:00


In [3]:
from huggingface_hub import login

# Apna Hugging Face ka Read Token yahan paste karein (Jo settings se banaya tha)
# Yeh aap se token mangega, paste kar ke enter daba dein.
login()

In [4]:
!pip install -q langchain langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [5]:
!pip install sentence-transformers faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 32.1 MB/s eta 0:00:00


In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. PDF Load karein (Apna sahi file name yahan likhein)
pdf_path = "/content/drive/MyDrive/Computer Science Short Notes Class 9th (1).pdf"
loader = PyPDFLoader(pdf_path)
docs = loader.load()

# 2. Text ko chunks mein divide karein
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(docs)

print(f" PDF successfully parsed into {len(chunks)} chunks!")

/tmp/ipykernel_674/3952327356.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


✅ PDF successfully parsed into 160 chunks!


In [7]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Creating FAISS Vector Database... (Is mein 1 minute lag sakta hai)")

# 1. Embedding model load karein
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. FAISS Index banayein
vector_db = FAISS.from_documents(chunks, embeddings)

print(" FAISS Vector Database is ready for Semantic Search!")

⏳ Creating FAISS Vector Database... (Is mein 1 minute lag sakta hai)


/tmp/ipykernel_674/3442856723.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🎯 FAISS Vector Database is ready for Semantic Search!


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

print("⏳ Loading Llama-3-8B-Instruct in 4-bit... (Is mein 2-3 mins lag sakte hain)")

# 1. 4-bit configuration set karein taake GPU crash na ho
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 2. Tokenizer load karein
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 3. Model load karein
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print(" BOOM! Llama-3 is successfully loaded on your T4 GPU!")

⏳ Loading Llama-3-8B-Instruct in 4-bit... (Is mein 2-3 mins lag sakte hain)


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

🔥 BOOM! Llama-3 is successfully loaded on your T4 GPU!


In [ ]:
def ask_bot(user_question):
    global llama_chat_history

    # 1. History extraction
    history_str = ""
    for msg in llama_chat_history.messages:
        role = "User" if msg.type == "human" else "Assistant"
        history_str += f"{role}: {msg.content}\n"

    # 2. Query Reformulation (if history exists)
    search_query = user_question
    if len(llama_chat_history.messages) > 0:
        reformulate_prompt = (
            f"Given the following chat history and a follow-up question, "
            f"rephrase it to be a standalone search query.\n\n"
            f"Chat History:\n{history_str}\n"
            f"Follow-up Question: {user_question}\n"
            f"Standalone Question:"
        )
        ref_inputs = tokenizer(reformulate_prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            ref_outputs = model.generate(**ref_inputs, max_new_tokens=50, temperature=0.1)
        ref_response = ref_outputs[0][ref_inputs["input_ids"].shape[-1]:]
        search_query = tokenizer.decode(ref_response, skip_special_tokens=True).strip()
        print(f" [System Refined Query]: {search_query}\n")

    # 3. FAISS Retrieval
    retrieved_docs = vector_db.similarity_search(search_query, k=2)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    # 4.   System PROMPT HERE 
    system_prompt = (
        "You are an expert Class 9 Computer Science tutor. "
        "Answer the user's question based ONLY on the provided Context and Chat History.\n"
        "STRICT RULES:\n"
        "1. If the user asks 'What is X?' or asks for a general concept, give ONLY the basic definition/explanation. Do NOT list its types, categories, or components unless explicitly asked.\n"
        "2. If the user asks a follow-up question like 'What are its types?', then look at the history, identify X, and list its types or components clearly.\n"
        "3. Do not repeat information that you already provided in the chat history.\n\n"
        f"--- CONTEXT ---\n{context}\n"
        f"--- CHAT HISTORY ---\n{history_str}"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_question}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.2, # Lower temperature makes it follow instructions strictly
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id
        )

    response = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(response, skip_special_tokens=True).strip()

    llama_chat_history.add_user_message(user_question)
    llama_chat_history.add_ai_message(answer)

    print(f" USER: {user_question}\n")
    print(f" Llama-3:\n{answer}")
    print("="*60)

In [23]:

llama_chat_history.clear()

# Test 1
ask_bot("What Are Transmission Mode?")

# Test 2 (Memory Test)
ask_bot("What are the types of it?")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 USER: What Are Transmission Mode?

 Llama-3:
Network communication modes describe how data is transmitted between devices.


[transformers] Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 [System Refined Query]: What are the different types of transmission modes in network communication?

Rephrased Search Query: "Types of transmission modes in network communication" or "Network transmission modes" or "Types of network communication modes" or "Network communication modes types" or

 USER: What are the types of it?

 Llama-3:
Based on our previous conversation, I understand that you're referring to Transmission Modes. According to our previous discussion, there are three primary types of transmission modes:

1. **Simplex Communication**: In this mode, data is transmitted in one direction only, from the sender to the receiver.
2. **Half-Duplex Communication**: In this mode, data is transmitted in both directions, but not simultaneously. The sender and receiver take turns transmitting and receiving data.
3. **Full-Duplex Communication**: In this mode, data is transmitted in both directions simultaneously, allowing for simultaneous sending and receiving of data.

These are 